# ___Phylogenetic traversal of ACEs___
---------------------------------------------------

In [2]:
print(R.version$version.string)

[1] "R version 4.5.2 (2025-10-31 ucrt)"


In [3]:
suppressPackageStartupMessages({
    library("ape")
    library("phytools")
    library("corHMM")
})

In [6]:
STATES <- read.csv("../../data/chapter2/FREDv3subset/finalized_states_395_species.csv", stringsAsFactors = TRUE)[, c("binominal", "state")] # finalized mycorrhizal states
COLLAB_AXIS <- read.csv("../../data/chapter2/FREDv3subset/collab_ord1_species_avgs_SRL_RD.csv", stringsAsFactors = TRUE) # first order species averaged RD and SRL values
MERGED <- merge(x = STATES, y = COLLAB_AXIS, by = "binominal") # merge the two datasets into one, based on the binominal names
stopifnot(nrow(MERGED)==395)

PHYLOGENY <- ape::multi2di(ape::read.tree("../../data/chapter2/uphylomaker/FRED_subset_collab_395sp.tre")) # phylogenetic tree created for the 395 species using U.PhyloMaker
stopifnot(length(PHYLOGENY$tip.label)==395)

# MERGED contains spaces in the binominal names - replace that with underscores; and '/' in mycorrhizal states that need to be removed
data <- data.frame(binominal = gsub(MERGED$binominal, pattern = ' ', replacement = '_'), RD = MERGED$F00679, SRL = MERGED$F00727, myco = gsub(x = MERGED$state, pattern = '/', replacement = ''))
matched_row_indices <- match(PHYLOGENY$tip.label, data$binominal)
stopifnot(all(data$binominal[matched_row_indices] == PHYLOGENY$tip.label))
data <- data[matched_row_indices, ] # reorder the dataset to match the species order in the phylogeny
stopifnot(all(data$binominal == PHYLOGENY$tip.label))
stopifnot(length(unique(data$binominal)) == length(data$binominal))

In [7]:
# FOR CONVENIRNCE
RD <- setNames(object = data$RD, nm = data$binominal)
SRL <- setNames(object = data$SRL, nm = data$binominal)
STATES <- setNames(object = data$myco, nm = data$binominal)

In [8]:
# PLOT THE PHYLOGENY AND NODE & TIP NUMBERS, IT'LL HELP IN TRACING DOWN CHANGES AND SHIFTS IN TRAITS
par(mar=c(0, 0, 0, 0))
png(filename = "../../plots/FRED_subset_collab_395sp_nodes_n_tips.png", width = 18000, height = 18000, units = "px", res = 300)
phytools::plotTree(tree = PHYLOGENY, ftype = "i", fsize = 1.2, type = "fan", lwd = 1, offset = 4)
phytools::labelnodes(text = 1:(PHYLOGENY$Nnode + length(PHYLOGENY$tip.label)), node = 1:(PHYLOGENY$Nnode + length(PHYLOGENY$tip.label)), cex = 1, interactive = FALSE)
dev.off()

agg_record_1770286312 
                    2

In [ ]:
#-----------------------------------------------
# ACE OF DISCRETE CATEGORICAL TRAITS
#-----------------------------------------------

states_rrace <- phytools::rerootingMethod(tree = PHYLOGENY, x = STATES, model = "ER")
states_corhmmace <- corHMM::corHMM(phy = PHYLOGENY, data = data[, c("binominal", "myco")], model = "ER", node.states = "marginal", rate.cat = 1)

#--------------------------------
# ACE OF CONTINUOUS TRAITS
#--------------------------------

srl_fastanc <- phytools::fastAnc(tree = PHYLOGENY, x = SRL)

In [ ]:
# reconstructed ancestral states of the internal nodes, with node numbers as names in the named vector
setNames(colnames(states_rrace$marginal.anc)[apply(states_rrace$marginal.anc, MARGIN = 1, FUN=which.max)], nm = rownames(states_rrace$marginal.anc))


# WHAT WE NEED HERE IS A DICTIONARY OF WHICH NODES DESCEND FROM WHICH NODES, SO WE CAN LOOK UP THE STATE TRANSITIONS AND CONTINUOUS TRAIT CHANGES
# http://www.phytools.org/eqg/Exercise_3.2/
# By convention, the tips of the tree are numbered 1 through n for n tips; and the nodes are numbered n + 1 through n + m for m nodes
# the matrix edge contains the beginning and ending node number for all the nodes and tips in the tree.
PHYLOGENY$edge

phyedges <- as.data.frame(PHYLOGENY$edge)
colnames(phyedges) <- c("from", "to")

for (i in 1:nrow(phyedges)) {
    print(srl_fastanc[as.character(phyedges[i, ][, "from"])]) # - srl_fastanc[phyedges[i, ][, "to"]])
}

